# Continuous Optimisation

## 0. Imports

In [ ]:
# class generation
import numpy as np
from numpy import linalg as LA
from typing import Tuple, List, Dict, Optional, Sequence, Callable
import time
# plots & visualization
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SEED = 2026

## 1. Structural Mechanics

### 1.1. Class & Method Definition

In [ ]:
class StructuralMechanics:
    """
    Define the objective function for the gradient descent of the structural mechanics problem.
    """
    
    def __init__(self, n: int = 20):
        # check whether u is in R^n
        assert n > 0, f"n = {n} is not a positive integer."

        self.n: int = n
        self.K: np.ndarray = np.zeros((self.n, self.n))
        self.b: np.ndarray = np.zeros(self.n)

        # b = (0 0 ... 0 1)
        self.b[-1] = 1

        # K = (  2 -1  0 ... 0 )
        #     ( -1  2 -1 ... 0 )
        #     (  0 -1  .     0 )
        #     (  .      .    . )
        #     (  .        . -1 )
        #     (  0  ... 0 -1 2 )
        for i in range(self.n):
            self.K[i, i] = 2
            if i > 0:
                self.K[i - 1, i] = -1
            if i < self.n - 1:
                self.K[i + 1, i] = -1
            if i == self.n - 1:
                self.K[i, i] = 1

        eigenvalues = LA.eigvalsh(self.K)
        self.mu: float = float(min(eigenvalues))
        self.eta: float = float(max(eigenvalues))
        self.max_alpha: float = float(2 / self.eta) # upper bound for alpha which guarantees gradient descent (prop 3.3)
        self.kappa: float = float(max(eigenvalues)/min(eigenvalues)) # condition number: k = L / mu
        
    def _objective(self, u: np.ndarray) -> float:
        """Compute the objective of the function."""
        # E(u) = 1/2 * u^T * K * u - b^T * u
        return 0.5 * u.T @ self.K @ u - self.b.T @ u
    
    def _gradient(self, u: np.ndarray) -> np.ndarray:
        """Compute the gradient of the function."""
        # grad E(u) = K * u - b
        return self.K @ u - self.b

    def _hessian(self) -> np.ndarray:
        """Compute the Hessian of the function."""   
        # hessian E(u) = K     
        return self.K.copy()

    def gradient_descent(
            self, 
            u_0: np.ndarray, 
            alpha: float = None, 
            epsilon: float = 1e-8, 
            max_iter: int = 10000,
            track_history: bool = False  # for plotting purposes
            ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Fixed Step Size
        
        Args:
            u_0 (np.ndarray): initial point
            alpha (float): step size
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """ 
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        
        if alpha is None: alpha = float(self.max_alpha / 2)
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        assert alpha < self.max_alpha, f"alpha = {alpha} does not guarantee gradient descent, try alpha < {self.max_alpha}."
        
        u_k = u_0.copy()

        # history tracking
        if track_history:
            hist_obj = []
            hist_grad_norm = []

        for k in range(max_iter):

            grad_k = self._gradient(u_k)
            grad_norm = LA.norm(grad_k)

            if track_history:
                hist_obj.append(self._objective(u_k))
                hist_grad_norm.append(float(grad_norm))
            
            if grad_norm < epsilon:
                if track_history:
                    return u_k, k, self._objective(u_k), {'obj': hist_obj, 'grad': hist_grad_norm}
                return u_k, k, self._objective(u_k)

            u_k -= alpha * grad_k
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def optimal_step(self, 
        u_k: np.ndarray, 
        d_k: np.ndarray
        ) -> float:
        """Optimal choice of alpha for quadratic functions in given by alpha = -(grad E(u) * d) / (d^T * K * d).

        Args:
            u_k (np.ndarray): current point
            d_k (np.ndarray): descent direction

        Returns:
            float: desired alpha
        """
        grad = self._gradient(u_k)
        numerator = - grad @ d_k
        denominator = d_k.T @ self.K @ d_k
        if abs(denominator) < 1e-12:
            return 0.0
        return min(float(numerator / denominator), self.max_alpha)

    def GD_exact_line_search(self,
        u_0: np.ndarray,
        epsilon: float = 1e-8,
        max_iter: int = 10000,
        track_history: bool = False  # for plotting purposes
        ) -> Tuple[np.ndarray, int, float]:    
        """Gradient Descent with Exact Line Search

        Args:
            u_0 (np.ndarray): initial point
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.
            track_history (bool, optional): Track obj, grad, and iterate history for plotting. Defaults to False.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """        
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        u_k = u_0.copy()

        # history tracking
        if track_history:
            hist_obj = []
            hist_grad_norm = []
            hist_alpha = []

        prev_obj = float('inf')
        stagnation_count = 0  # ADD TRACKING
        STAGNATION_THRESHOLD = 50  # NEW CONSTANT

        for k in range(max_iter):
            grad_k = self._gradient(u_k)
            grad_norm = LA.norm(grad_k)
            alpha_k = self.optimal_step(u_k, -grad_k)

            if track_history:
                hist_obj.append(self._objective(u_k))
                hist_grad_norm.append(float(grad_norm))
                hist_alpha.append(alpha_k)
            
            if grad_norm < epsilon:
                if track_history:
                    return u_k, k, self._objective(u_k), {'obj': hist_obj, 'grad': hist_grad_norm, 'alpha': hist_alpha}
                return u_k, k, self._objective(u_k)

            u_k -= alpha_k * grad_k

            # STAGNATION CHECK
            curr_obj = self._objective(u_k)
            if abs(curr_obj - prev_obj) < 1e-12:
                stagnation_count += 1
                if stagnation_count > STAGNATION_THRESHOLD:
                    # print(f"  Warning: Stagnated after {k} iterations, ||grad|| = {grad_norm:.2e}")
                    if track_history:
                        return u_k, k, curr_obj, {'obj': hist_obj, 'grad': hist_grad_norm, 'alpha': hist_alpha}
                    return u_k, k, curr_obj  # accepts current solution
            else:
                stagnation_count = 0
            prev_obj = curr_obj
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def armijo_condition(self,
        u: np.ndarray,
        d: np.ndarray,
        alpha: float,
        c: float = 1e-4 # common value for this parameter (p. 42 in the Lecture notes)
        ) -> bool:
        """Check E(u + alpha * d) <= E(u) + c* alpha * grad E(u)^T * d.

        Args:
            u (np.ndarray): Current position.
            d (np.ndarray): Descent direction.
            alpha (float): Step size.
            c (float, optional): Constant coefficient. Defaults to 1e-4.

        Returns:
            bool: Is Armijo satisfied?
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        assert alpha < self.max_alpha, f"alpha = {alpha} does not guarantee gradient descent, try alpha < {self.max_alpha}."
        step = self._objective(u + alpha * d)
        current = self._objective(u)
        change = c * alpha * self._gradient(u).T @ d
        return (step <= current + change)

    def backtracking(self,
        u_k: np.ndarray,
        d_k: np.ndarray,
        alpha_0: float = None,
        rho: float = 0.5, 
        c: float = 1e-4,  # common value for this parameter (p. 42 in the Lecture notes)
        max_iter: int = 10000
        ) -> float:
        """Backtracking line search for sufficiently optimal alpha.

        Args:
            u_k (np.ndarray): Current point.
            d_k (np.ndarray): Descent direcion.
            alpha_0 (float, optional): Initial alpha. Defaults to 1/eta.
            rho (float, optional): Constant rho. Defaults to 0.5.
            c (float, optional): Constant c. Defaults to 1e-4.
            max_iter (int, optional): Maximum iterations (primarily for debugging). Defaults to 10000.

        Returns:
            float: New sufficient alpha.
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        
        if alpha_0 is None: alpha_0 = float(self.max_alpha / 2)
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert alpha_0 < self.max_alpha, f"alpha = {alpha_0} does not guarantee gradient descent, try alpha < {self.max_alpha}."
        
        alpha_new = alpha_0
        
        for _ in range(max_iter):
            if self.armijo_condition(u_k, d_k, alpha_new, c):
                return float(alpha_new)
            alpha_new *= rho

        return float(alpha_new)

    def GD_inexact_line_search(self,
        u_0: np.ndarray,
        alpha_0: float = None,
        rho: float = 0.5,
        c: float = 1e-4,
        epsilon: float = 1e-8,
        max_iter: int = 10000,
        track_history: bool = False  # for plotting purposes
        ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Inexact Line Search

        Args:
            u_0 (np.ndarray): Starting point.
            alpha_0 (float, optional): Initial alpha. Defaults to 1/eta.
            rho (float, optional): Constant rho. Defaults to 0.5.
            c (float, optional): Constant c. Defaults to 1e-4.
            epsilon (float, optional): Error margin. Defaults to 1e-8.
            max_iter (int, optional): Maximum number of iterations (primarily for debugging). Defaults to 10000.
            track_history (bool, optional): Track obj, grad, and iterate history for plotting. Defaults to False.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """     
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        
        if alpha_0 is None: alpha_0 = float(self.max_alpha / 2)
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert alpha_0 < self.max_alpha, f"alpha = {alpha_0} does not guarantee gradient descent, try alpha < {self.max_alpha}."
        
        u_k = u_0.copy()
        alpha_k = alpha_0

        # history tracking
        if track_history:
            hist_obj = []
            hist_grad_norm = []
            hist_alpha = []

        prev_obj = float('inf')
        stagnation_count = 0  # track stagnation
        STAGNATION_THRESHOLD = 50  # threshold constant

        for k in range(max_iter):
            grad_k = self._gradient(u_k) # = - d_k
            grad_norm = LA.norm(grad_k)
            alpha_k = self.backtracking(u_k=u_k, d_k=(-grad_k), alpha_0=alpha_k, rho=rho, c=c)

            if track_history:
                hist_obj.append(self._objective(u_k))
                hist_grad_norm.append(float(grad_norm))
                hist_alpha.append(alpha_k)
            
            if grad_norm < epsilon:
                if track_history:
                    return u_k, k, self._objective(u_k), {'obj': hist_obj, 'grad': hist_grad_norm, 'alpha': hist_alpha}
                return u_k, k, self._objective(u_k)

            u_k -= alpha_k * grad_k

            # stagnation check
            curr_obj = self._objective(u_k)
            if abs(curr_obj - prev_obj) < 1e-12:
                stagnation_count += 1
                if stagnation_count > STAGNATION_THRESHOLD:
                    # print(f"  Warning: Stagnated after {k} iterations, ||grad|| = {grad_norm:.2e}")
                    if track_history:
                        return u_k, k, curr_obj, {'obj': hist_obj, 'grad': hist_grad_norm, 'alpha': hist_alpha}
                    return u_k, k, curr_obj  # accept current solution
            else:
                stagnation_count = 0
            prev_obj = curr_obj
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def heavy_ball(
        self,
        u_0: np.ndarray,
        alpha: float,
        beta: float,
        epsilon: float = 1e-8,
        max_iter: int = 10000,
        track_history: bool = False
        ) -> Tuple[np.ndarray, int, float]:
        """Heavy-ball method as presented by Algorithm 4.1 in the Lecture Notes.

        Args:
            u_0 (np.ndarray): Initial point.
            alpha (float): Step size.
            beta (float): Momentum parameter.
            epsilon (float, optional): Error margin. Defaults to 1e-8.
            max_iter (int, optional): Maximum number of iterations. Defaults to 10000.
            track_history (bool, optional): History tracking for data and plotting. Defaults to False.

        Raises:
            ValueError: Objective or gradeint becomes non-finite.
            RuntimeError: Maximum number of iterations reached.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """        
        assert alpha > 0
        assert 0 <= beta < 1
        assert epsilon > 0 and max_iter > 0

        x_k = np.asarray(u_0, dtype=float).copy()
        x_k_prev = x_k.copy()  # x_{-1} = x_0
        history = {'obj': [], 'grad': []}

        for k in range(max_iter + 1):
            grad_k = self._gradient(x_k)
            grad_norm = LA.norm(grad_k)
            obj_k = self._objective(x_k)

            if not np.isfinite(obj_k) or not np.isfinite(grad_norm):
                raise ValueError("Objective or gradient became non-finite")

            if track_history:
                history['obj'].append(float(obj_k))
                history['grad'].append(float(grad_norm))

            if grad_norm < epsilon:
                result = (x_k, k, obj_k)
                return (*result, history) if track_history else result

            if k == max_iter:
                break

            x_k_next = x_k - alpha * grad_k + beta * (x_k - x_k_prev)
            x_k_prev, x_k = x_k, x_k_next

        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def nesterov(
        self,
        u_0: np.ndarray,
        alpha: float,
        beta: float,
        epsilon: float = 1e-8,
        max_iter: int = 10000,
        track_history: bool = False
        ) -> Tuple[np.ndarray, int, float]:
        """Nesterov's accelerated gradient method as presented by Algorithm 4.2 in the Lecture Notes.

        Args:
            u_0 (np.ndarray): Initial point.
            alpha (float): Step size.
            beta (float): Momentum parameter.
            epsilon (float, optional): Error margin. Defaults to 1e-8.
            max_iter (int, optional): Maximum number of iterations. Defaults to 10000.
            track_history (bool, optional): History tracking for data and plotting. Defaults to False.

        Raises:
            ValueError: Objective or gradeint becomes non-finite.
            ValueError: Extrapolated gradient became non-finite.
            RuntimeError: Maximum number of iterations reached.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """
        assert alpha > 0
        assert 0 <= beta < 1
        assert epsilon > 0 and max_iter > 0

        x_k = np.asarray(u_0, dtype=float).copy()
        x_k_prev = x_k.copy()  # x_{-1} = x_{0}
        history = {'obj': [], 'grad': []}

        for k in range(max_iter + 1):
            grad_k = self._gradient(x_k)
            grad_norm = LA.norm(grad_k)
            obj_k = self._objective(x_k)

            if not np.isfinite(obj_k) or not np.isfinite(grad_norm):
                raise ValueError("Objective or gradient became non-finite")

            if track_history:
                history['obj'].append(float(obj_k))
                history['grad'].append(float(grad_norm))

            if grad_norm < epsilon:
                result = (x_k, k, obj_k)
                return (*result, history) if track_history else result

            if k == max_iter:
                break

            # evaluate the update gradient at the extrapolated point
            y_k = x_k + beta * (x_k - x_k_prev)
            grad_y = self._gradient(y_k)

            if not np.all(np.isfinite(grad_y)):
                raise ValueError("Extrapolated gradient became non-finite")

            x_k_next = y_k - alpha * grad_y
            x_k_prev, x_k = x_k, x_k_next

        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def newton(
        self,
        u_0: np.ndarray,
        epsilon: float = 1e-8,
        max_iter: int = 10000,
        track_history: bool = False
        ) -> Tuple[np.ndarray, int, float]:
        """Newton's method as presented by Algorithm 4.3 in the Lecture Notes.

        Args:
            u_0 (np.ndarray): Initial position.
            epsilon (float, optional): Error margin. Defaults to 1e-8.
            max_iter (int, optional): Maximum number of iterations. Defaults to 10000.
            track_history (bool, optional): History tracking for data and plotting. Defaults to False.

        Raises:
            ValueError: Objective or gradeint becomes non-finite.
            RuntimeError: Maximum number of iterations reached.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """
        assert epsilon > 0,  f"epsilon = {epsilon} is not > 0."
        assert max_iter > 0, f"max iterations = {max_iter} is not > 0."

        x_k = np.asarray(u_0, dtype=float).copy()
        history = {'obj': [], 'grad': []}

        for k in range(max_iter + 1):
            grad_k = self._gradient(x_k)
            hessian = self._hessian().copy()
            d_k = LA.solve(hessian, -grad_k)
            grad_norm = LA.norm(grad_k)
            obj_k = self._objective(x_k)

            if not np.isfinite(obj_k) or not np.isfinite(grad_norm):
                raise ValueError("Objective or gradient became non-finite")

            if track_history:
                history['obj'].append(float(obj_k))
                history['grad'].append(float(grad_norm))

            if grad_norm < epsilon:
                result = (x_k, k, obj_k)
                return (*result, history) if track_history else result

            if k == max_iter:
                break

            # update x_k
            x_k += d_k

        raise RuntimeError(f"Did not converge within {max_iter} iterations")

### 1.2. Verification for the Correctness of Implementation

In [ ]:
###### DOCTEST SUITE #####
if __name__ == "__main__":
    
    def run_doctests():
        """Run doctests to verify correctness of implementation for all optimization methods."""
        
        print("~~" * 60)
        print("Running Structural Mechanics Doctests")
        print("~~" * 60)

        ### Initialization ###
        
        # Test 1: Basic initialization
        print("\n[Test 1] Initialization and matrix construction...")
        sm = StructuralMechanics(20)
        assert sm.n == 20, "n should be 20"
        assert sm.b[-1] == 1, "Last element of b should be 1"
        assert sm.b[:-1].sum() == 0, "First n - 1 elements of b should be 0"
        print("🖒 Matrix K and vector b initialized correctly")
            
        # Test 2: K matrix properties
        print("\n[Test 2] Stiffness matrix properties...")
        K = sm.K
        diag = np.diag(K)
        assert np.allclose(diag[:-1], 2), "First n-1 diagonal elements should be 2"
        assert np.isclose(diag[-1], 1), f"Last diagonal element should be 1, got {diag[-1]}"
        off_diag = np.diag(K, 1)
        assert np.allclose(off_diag, -1), "Second biggest diagonal should be -1"
        assert np.allclose(K.T, K), "Matrix should be symmetric"
        print("🖒 Tridiagonal structure verified (with boundary condition)")
        
        # Test 3: Objective function calculation
        print("\n[Test 3] Objective function...")
        u_test = np.ones(20)
        manual_energy = 0.5 * u_test.T @ sm.K @ u_test - sm.b.T @ u_test
        assert np.isclose(sm._objective(u_test), manual_energy), "Energy calculation mismatch"
        print(f"🖒 Objective = {sm._objective(u_test):.6f}")
        
        # Test 4: Gradient calculation
        print("\n[Test 4] Gradient computation...")
        grad_manual = K @ u_test - sm.b
        assert np.allclose(sm._gradient(u_test), grad_manual), "Gradient mismatch"
        print("🖒 Gradient verified analytically")
        
        # Test 5: Eigenvalue properties (convexity check)
        print("\n[Test 5] Convexity via eigenvalues...")
        assert sm.mu > 0, "Problem should be strictly convex (mu > 0)"
        print(f"  lambda_min = {sm.mu:.6f}, lambda_max = {sm.eta:.6f}, kappa = {sm.kappa:.2f}")
        print("🖒 Problem is conditioned and convex")

        ### Gradient Descent Methods ###
        
        # Test 6: Fixed-step gradient descent
        print("\n[Test 6] Fixed-step gradient descent...")
        u0_small = np.zeros(10)
        sm_small = StructuralMechanics(10)
        optimal_alpha = 2 / (sm_small.eta + sm_small.mu)
        u_fixed, iter_fixed, obj_fixed = sm_small.gradient_descent(
            u0_small, alpha=optimal_alpha, epsilon=1e-10
        )
        print(f"  Converged in {iter_fixed} iterations, final obj = {obj_fixed:.8f}")
        print("🖒 Fixed-step descent works")
        
        # Test 7: Exact line search
        print("\n[Test 7] Exact line search...")
        u_exact, iter_exact, obj_exact = sm_small.GD_exact_line_search(
            u_0=u0_small, epsilon=1e-10
        )
        print(f"  Converged in {iter_exact} iterations, final obj = {obj_exact:.8f}")
        print("🖒 Exact line search converged")
        
        # Test 8: Inexact line search (backtracking)
        print("\n[Test 8] Inexact line search (backtracking)...")
        u_inexact, iter_inexact, obj_inexact = sm_small.GD_inexact_line_search(
            u_0=u0_small, epsilon=1e-10
        )
        print(f"  Converged in {iter_inexact} iterations, final obj = {obj_inexact:.8f}")
        print("🖒 Backtracking line search worked")
        
        ### ADVANCED METHODS ###
        
        # Test 9: Heavy-ball method
        print("\n[Test 9] Heavy-ball momentum method...")
        # Polyak's optimal parameters for strongly convex quadratics
        kappa_small = sm_small.kappa
        beta_hb = ((np.sqrt(kappa_small) - 1) / (np.sqrt(kappa_small) + 1)) ** 2
        alpha_hb = 4 / ((np.sqrt(sm_small.eta) + np.sqrt(sm_small.mu)) ** 2)
        
        u_hb, iter_hb, obj_hb = sm_small.heavy_ball(
            u_0=u0_small, alpha=alpha_hb, beta=beta_hb, epsilon=1e-10
        )
        print(f"  beta = {beta_hb:.4f}, alpha = {alpha_hb:.6f}")
        print(f"  Converged in {iter_hb} iterations, final obj = {obj_hb:.8f}")
        print("🖒 Heavy-ball method converged")
        
        # Test 10: Nesterov's accelerated gradient
        print("\n[Test 10] Nesterov's accelerated gradient...")
        # Optimal Nesterov parameters (similar to Heavy-ball for quadratic)
        beta_nest = ((np.sqrt(kappa_small) - 1) / (np.sqrt(kappa_small) + 1)) ** 2
        alpha_nest = 1 / sm_small.eta  # Conservative step size
        
        u_nest, iter_nest, obj_nest = sm_small.nesterov(
            u_0=u0_small, alpha=alpha_nest, beta=beta_nest, epsilon=1e-10
        )
        print(f"  beta = {beta_nest:.4f}, alpha = {alpha_nest:.6f}")
        print(f"  Converged in {iter_nest} iterations, final obj = {obj_nest:.8f}")
        print("🖒 Nesterov acceleration worked")
        
        # Test 11: Newton's method
        print("\n[Test 11] Newton's method...")
        u_newton, iter_newton, obj_newton = sm_small.newton(
            u_0=u0_small, epsilon=1e-10
        )
        print(f"  Converged in {iter_newton} iterations, final obj = {obj_newton:.8f}")
        print("🖒 Newton's method converged (quadratic convergence expected)")
        
        # Test 12: Comparison of method performance
        print("\n[Test 12] Full method comparison...")
        print(f"\n{'Method':<30} {'Iterations':>12} {'Final Obj':>18}")
        print(f"{'-'*30} {'-'*12} {'-'*18}")
        print(f"{'Fixed Step':<30} {iter_fixed:>12} {obj_fixed:>18.8f}")
        print(f"{'Exact Line Search':<30} {iter_exact:>12} {obj_exact:>18.8f}")
        print(f"{'Inexact (Backtracking)':<30} {iter_inexact:>12} {obj_inexact:>18.8f}")
        print(f"{'Heavy-Ball':<30} {iter_hb:>12} {obj_hb:>18.8f}")
        print(f"{'Nesterov Accelerated':<30} {iter_nest:>12} {obj_nest:>18.8f}")
        print(f"{'Newton':<30} {iter_newton:>12} {obj_newton:>18.8f}")
        
        # Test 13: Convergence to analytical solution
        print("\n[Test 13] Analytical solution verification...")
        u_analytical = LA.solve(sm_small.K, sm_small.b)
        
        methods_to_check = [
            ('Fixed', u_fixed),
            ('Exact LS', u_exact),
            ('Backtracking', u_inexact),
            ('Heavy-Ball', u_hb),
            ('Nesterov', u_nest),
            ('Newton', u_newton)
        ]
        
        for method_name, u_numerical in methods_to_check:
            error = LA.norm(u_numerical - u_analytical)
            status = "(🖒^_^)🖒" if error < 1e-4 else "ー═┻┳︻▄ξ(╬ಠ益ಠ)ξ▄︻┻┳═一"
            print(f"    {method_name:<15} ||u_num - u_exact|| = {error:.2e} {status}")
            assert error < 1e-4, f"{method_name} convergence error too large: {error}"
        
        print("🖒 All methods converge to analytical minimum")
                
        # Test 14: Quadratic convergence of Newton (property, specific to Newton's method)
        print("\n[Test 14] Newton's quadratic convergence check...")
        if iter_newton > 1:
            # For quadratic problems, Newton should converge in very few iterations.
            print(f"  Newton converged in {iter_newton} iterations")
            print(f"  Expected: O(1) iterations for quadratic objectives")
            print(f"🖒 Newton's quadratic convergence {"TRUE" if (iter_newton <= 20) else "FALSE"}")
        else:
            print(f"  Newton converged in {iter_newton} iteration(s)")
            print(f"🖒 Excellent convergence (very fast)")
        
        # Test 15: Momentum methods acceleration verification
        print("\n[Test 15] Acceleration vs standard GD...")
        speedup_hb = iter_exact / iter_hb if iter_hb > 0 else float('inf')
        speedup_nest = iter_exact / iter_nest if iter_nest > 0 else float('inf')
        
        print(f"  Heavy-ball speedup: {speedup_hb:.2f}x vs fixed step")
        print(f"  Nesterov speedup:   {speedup_nest:.2f}x vs fixed step")
        
        if speedup_hb > 1 or speedup_nest > 1:
            print("🖒 Momentum methods show acceleration")
        else:
            print("NB: Speedup may vary depending on condition number")
        
        print("\n" + "~~" * 60)
        print("All tests passed! (🖒^_^)🖒")
        print("~~" * 60)
    
    run_doctests()

### 1.3. Numerical Experiments

In [ ]:
def GD_method_comparison(n_values: Optional[Sequence[int]] = None, 
                         max_iter: int = 600000, 
                         epsilon: float = 1e-8,
                         history_n: Optional[int] = None, 
                         annotate_bars: bool = True):
    """Sweep the line-search experiments over several problem sizes n.
    The problem is deterministic, so no multiple attempts necessary (in contrast to SGD).

    Args:
        n_values (sequence[int], optional): Problem sizes to sweep. Defaults to 10, 20, ..., 100.
        max_iter (int, optional): Maximum iterations per run. Defaults to 600000.
        epsilon (float, optional): Gradient-norm tolerance. Defaults to 1e-8.
        history_n (int, optional): Convergence panels (c) / (d) based on n = history_n. Defaults to max(n_values).
        annotate_bars (bool, optional): Label the bars in panel (b) with their values. Defaults to True.
    """
    if n_values is None:
        n_values = list(range(10, 101, 10))
    n_values = list(n_values)
    if history_n is None:
        history_n = max(n_values)

    METHOD_COLORS = {'Fixed': 'tab:blue', 'Exact': 'tab:green', 'Backtracking': 'tab:orange'}
    METHODS = list(METHOD_COLORS)

    results = {n: {} for n in n_values}       # n -> method -> iters (scalar)
    alpha_sweep = {n: [] for n in n_values}   # n -> [(ratio, iters, converged)]
    kappa_by_n, alpha_opt_by_n = {}, {}
    histories_ref, sm_ref = {}, None

    print("~" * 67)
    print(f"Sweeping n = {n_values} | max_iter = {max_iter} | eps = {epsilon}")
    print("~" * 67)

    t0 = time.time()
    for n in n_values:
        sm = StructuralMechanics(n)
        u0 = np.zeros(n)
        alpha_opt = 2 / (sm.mu + sm.eta)

        kappa_by_n[n] = sm.kappa
        alpha_opt_by_n[n] = alpha_opt
        if n == history_n:
            sm_ref = sm

        track = (n == history_n) # track history only for the second panels
        runs = {
            'Exact': lambda: sm.GD_exact_line_search(u0, epsilon, max_iter, track_history=track),
            'Fixed': lambda: sm.gradient_descent(u0, alpha_opt, epsilon, max_iter, track_history=track),
            'Backtracking': lambda: sm.GD_inexact_line_search(u0, None, 0.5, 1e-4, epsilon, max_iter, track_history=track),
        }

        for m, run in runs.items():
            try:
                out = run()
                results[n][m] = min(int(out[1]), max_iter)
                if len(out) > 3 and out[3] is not None:
                    histories_ref[m] = out[3]
            except Exception:
                results[n][m] = max_iter   # diverged or surpassed iterations threshold

        # alpha sweep over n
        for ratio in np.linspace(0.1, 1.0, 10):
            try:
                _, k, _ = sm.gradient_descent(u0, ratio * sm.max_alpha, epsilon, max_iter)
                alpha_sweep[n].append((ratio, min(int(k), max_iter), True))
            except Exception:
                alpha_sweep[n].append((ratio, max_iter, False))

        print(f"  n={n:3d} | kappa={sm.kappa:9.1f} | "
              + " | ".join(f"{m}: {results[n][m]:7,d}" for m in METHODS)
              + f" | {time.time() - t0:6.1f}s")

    ### 4-PLOT VISUALIZATION ###
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    # ---- Plot 1: iterations vs alpha, one curve per n ----
    ax1 = axes[0, 0]
    cmap = plt.get_cmap('viridis')
    denom = max(len(n_values) - 1, 1)
    for idx, n in enumerate(n_values):
        xs = [d[0] for d in alpha_sweep[n]]
        ys = [d[1] for d in alpha_sweep[n]]
        ok = [d[2] for d in alpha_sweep[n]]
        ax1.plot(xs, ys, '-o', color=cmap(idx / denom), markersize=4, alpha=0.85, label=f'n={n}')
        dx = [x for x, o in zip(xs, ok) if not o]
        dy = [y for y, o in zip(ys, ok) if not o]
        if dx:
            ax1.plot(dx, dy, 'x', color='red', markersize=9, mew=2)
    ax1.plot([], [], 'x', color='red', markersize=9, mew=2, label='did not converge')
    ax1.axvline(0.5, color='blue', ls='--', lw=1, label=r'$\alpha=\alpha_{\max}/2$')
    ax1.axvline(alpha_opt_by_n[history_n] / sm_ref.max_alpha, color='black', ls=':', lw=1.5,
                label=rf'optimal $\alpha$ (n={history_n})')
    ax1.set_xlabel(r'$\alpha/\alpha_{\max}$'); ax1.set_ylabel('Iterations')
    ax1.set_title('(a) Step Size Influence Across n')
    ax1.set_yscale('log')
    ax1.legend(fontsize=8, ncol=2); ax1.grid(True, alpha=0.3, which='both')

    # ---- Plot 2: grouped bars, 3 methods x each n (deterministic -> one value per bar) ----
    ax2 = axes[0, 1]
    positions = np.arange(len(n_values))
    width = 0.25
    offsets = {'Fixed': -width, 'Exact': 0.0, 'Backtracking': width}

    for m in METHODS:
        vals = [results[n][m] for n in n_values]
        bars = ax2.bar(positions + offsets[m], vals, width, label=m,
                       color=METHOD_COLORS[m], alpha=0.85,
                       edgecolor='black', linewidth=0.8)
        if annotate_bars:
            for bar, val in zip(bars, vals):
                ax2.text(bar.get_x() + bar.get_width() / 2, val * 1.08, f'{val:,}',
                         ha='center', va='bottom', fontsize=6.5, rotation=90)

    ax2.set_xticks(positions)
    ax2.set_xticklabels([str(n) for n in n_values])
    ax2.set_xlabel('Problem size $n$'); ax2.set_ylabel('Iterations')
    ax2.set_title('(b) Iterations vs Problem Size')
    ax2.set_yscale('log')
    ax2.legend(fontsize=9, framealpha=0.9)
    ax2.grid(True, alpha=0.3, axis='y', which='both')
    ax2.set_axisbelow(True)

    # ---- Plot 3: objective convergence at history_n ----
    ax3 = axes[1, 0]
    any_history = any(h is not None and 'obj' in h for h in histories_ref.values())
    if any_history:
        u_analytical = LA.solve(sm_ref.K, sm_ref.b)
        obj_analytical = sm_ref._objective(u_analytical)
        max_len = 0
        for name, hist in histories_ref.items():
            if hist is not None and len(hist.get('obj', [])) > 1:
                obj_vals = hist['obj']
                ax3.semilogy(range(len(obj_vals)),
                             [max(abs(o - obj_analytical), 1e-16) for o in obj_vals],
                             label=name, linewidth=2, alpha=0.8)
                max_len = max(max_len, len(obj_vals))

        # Theoretical upper bound for fixed-step GD (Theorem 3.3)
        # f(x_k) - f* <= (1 - mu/L)^k * (f(u0) - f(u*)), for 0 < alpha < 2/L
        u_0 = np.zeros(sm_ref.n)
        obj_0 = sm_ref._objective(u_0)
        q = 1 - 1 / sm_ref.kappa
        obj_gap = obj_0 - obj_analytical

        k_vals = np.arange(1, max_len + 1)
        obj_bound = (q ** k_vals) * obj_gap

        ax3.semilogy(k_vals, obj_bound, ':', color='black', linewidth=2,
                     label=r'Thm 3.3 bound: $\left(\frac{\kappa - 1}{\kappa + 1}\right)^k (E_0 - E^*)$')
        ax3.set_xlabel('Iteration'); ax3.set_ylabel(r'$|E(u) - E^*|$')
        ax3.set_title(f'(c) Objective Convergence (n={history_n})')
        ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3, which='both')

        floor_val = 1e-16
        ax3.set_ylim(bottom=1e-8, top=max(obj_bound) * 1.5)
    else:
        ax3.text(0.5, 0.5, 'Add track_history=True\nto StructuralMechanics class\nfor convergence data',
                 ha='center', va='center', fontsize=12, transform=ax3.transAxes,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
        ax3.set_xlim(0, 1); ax3.set_ylim(0, 1); ax3.axis('off')

    # ---- Plot 4: gradient norm at history_n ----
    ax4 = axes[1, 1]
    any_grad = any(h is not None and 'grad' in h and len(h.get('grad', [])) > 0
                   for h in histories_ref.values())
    if any_grad:
        for name, hist in histories_ref.items():
            if hist is not None and len(hist.get('grad', [])) > 0:
                grad_vals = hist['grad']
                ax4.semilogy(range(len(grad_vals)), grad_vals, label=name, linewidth=2,
                             alpha=0.8)
        ax4.axhline(y=epsilon, color='gray', linestyle='--',
                    label=rf'$\varepsilon$={epsilon}', linewidth=1)
        ax4.set_xlabel('Iteration'); ax4.set_ylabel(r'$\|\nabla E(u_k)\|$')
        ax4.set_title(f'(d) Convergence of Gradient Norm (n={history_n})')
        ax4.legend(); ax4.grid(True, alpha=0.3, which='both')
    else:
        ax4.text(0.5, 0.5, 'Add track_history=True\nto StructuralMechanics class\nfor gradient data',
                 ha='center', va='center', fontsize=12, transform=ax4.transAxes,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
        ax4.set_xlim(0, 1); ax4.set_ylim(0, 1); ax4.axis('off')

    plt.tight_layout()
    fname = f'line_search_sweep_n{n_values[0]}-{n_values[-1]}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"\nSaved: {fname}")


# Run
if __name__ == "__main__":
    GD_method_comparison(n_values=range(10, 101, 10))

In [ ]:
def GD_method_comparison(n=20, max_iter=10000, epsilon=1e-8):
    """Complete experiments with 4 visualization panels (handles empty history).

    Args:
        n (int, optional): Size of the problem. Defaults to 20.
        max_iter (int, optional): Maximum number of iterations. Defaults to 10000.
        epsilon (_type_, optional): Error margin. Defaults to 1e-8.
    """
    sm = StructuralMechanics(n=n)
    print("~~" * 70)
    print(f"Line Search Experiments | n={n}, kappa={sm.kappa:.2f}")
    print("~~" * 70)
    
    ### EXPERIMENT 1: Alpha Sweep ###
    print("\nPart 1: Fixed-step gradient descent vs alpha")
    exp1_data = []
    for ratio in np.linspace(0.1, 0.9, 9):
        alpha = ratio * sm.max_alpha
        try:
            _, k, obj = sm.gradient_descent(np.zeros(n), alpha, epsilon, max_iter)
            exp1_data.append((ratio, k, True, obj))
            print(f"  alpha={ratio:.1f}*max: {k} iters ✔️")
        except:
            exp1_data.append((ratio, max_iter, False, float('inf')))
            print(f"  alpha={ratio:.1f}*max: diverged ❌️")
    
    ### EXPERIMENT 2: Methods with Tracked History ###
    print("\nPart 2: Method comparison + convergence curves")
    methods = []
    histories = {}
    
    # Fixed optimal step
    alpha_opt = 2/(sm.mu + sm.eta)
    try:
        u_f, k_f, obj_f, hist_f = sm.gradient_descent(np.zeros(n), alpha_opt, 
                                                     epsilon, max_iter, track_history=True)
        methods.append(('Fixed (optimal)', k_f, obj_f)); histories['Fixed'] = hist_f
        print(f"  Fixed (optimal): {k_f} iters, hist_len={len(hist_f['obj']) if hist_f else 0}")
    except:
        methods.append(('Fixed (optimal)', max_iter, float('inf'))); histories['Fixed'] = None
    
    # Exact line search
    try:
        u_e, k_e, obj_e, hist_e = sm.GD_exact_line_search(np.zeros(n), epsilon, 
                                                        max_iter, track_history=True)
        methods.append(('Exact Line Search', k_e, obj_e)); histories['Exact'] = hist_e
        print(f"  Exact Line Search: {k_e} iters, hist_len={len(hist_e['obj']) if hist_e else 0}")
    except:
        methods.append(('Exact Line Search', max_iter, float('inf'))); histories['Exact'] = None
    
    # Backtracking
    try:
        u_b, k_b, obj_b, hist_b = sm.GD_inexact_line_search(np.zeros(n), None, 
                                                          0.5, 1e-4, epsilon, max_iter, 
                                                          track_history=True)
        methods.append(('Backtracking', k_b, obj_b)); histories['Backtracking'] = hist_b
        print(f"  Backtracking: {k_b} iters, hist_len={len(hist_b['obj']) if hist_b else 0}")
    except:
        methods.append(('Backtracking', max_iter, float('inf'))); histories['Backtracking'] = None
    
    ### 4-PLOT VISUALIZATION ###
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Iterations vs alpha
    ax1 = axes[0, 0]
    ratios, iters, convs, objs = zip(*exp1_data)
    colors = ['green' if c else 'red' for c in convs]
    ax1.scatter(ratios, iters, c=colors, s=100, alpha=0.7)
    ax1.axvline(x=0.5, color='blue', linestyle='--', label=r'$\alpha = \alpha_\max / 2$')
    ax1.axvline(x=alpha_opt/sm.max_alpha, color='orange', linestyle='--', label='Optimal alpha')
    ax1.set_xlabel(r'$\alpha / \alpha_\max$'); ax1.set_ylabel('Iterations'); ax1.set_title('(a) Step Size Influence')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    
    # Plot 2: Bar chart comparison
    ax2 = axes[0, 1]
    names = [m[0] for m in methods]; ks = [m[1] for m in methods]
    bars = ax2.bar(names, ks, color=['blue','green','orange'], alpha=0.7)
    ax2.set_ylabel('Iterations'); ax2.set_title('(b) Method Comparison')
    for bar, val in zip(bars, ks): 
        if val < max_iter:  # Only label converged methods
            ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(val), ha='center')
    ax2.tick_params(axis='x', rotation=15); ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Objective convergence
    ax3 = axes[1, 0]
    any_history = any(h is not None and 'obj' in h for h in histories.values())
    if any_history:
        # Analytical minimiser and minimum value
        u_analytical = LA.solve(sm.K, sm.b)
        obj_analytical = sm._objective(u_analytical)
        max_len = 0
        for name, hist in histories.items():
            if hist is not None and len(hist.get('obj', [])) > 1:
                obj_vals = hist['obj']
                ax3.semilogy(range(len(obj_vals)),
                             [max(abs(o - obj_analytical), 1e-16) for o in obj_vals],
                             label=name, linewidth=2, alpha=0.8)
                max_len = max(max_len, len(obj_vals))

        # Theoretical upper bound for fixed-step GD (Theorem 3.3)
        # f(x_k) - f* <= (1 - mu/L)^k * (f(u0) - f(u*)), for 0 < alpha < 2/L
        u_0 = np.zeros(sm.n)
        obj_0 = sm._objective(u_0)
        q = 1 - 1 / sm.kappa # contraction factor
        obj_gap = obj_0 - obj_analytical

        k_vals = np.arange(1, max_len + 1) # bound undefined at k = 0
        obj_bound = (q ** k_vals) * obj_gap

        # Theorem 3.3 bound: geometric decay
        ax3.semilogy(k_vals, obj_bound, ':', color='black', linewidth=2,
                     label=r'Thm 3.3 bound: $\left(\frac{\kappa - 1}{\kappa + 1}\right)^k (f_0 - f^*)$')
        ax3.set_xlabel('Iteration'); ax3.set_ylabel(r'$|f(x) - f^*|$')
        ax3.set_title('(b) Objective Convergence')
        ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3, which='both')

        floor_val = 1e-16 # avoid cancellation resulting in 0
        data_min = min(
            min(max(abs(o - obj_analytical), floor_val) for o in h['obj'])
            for h in histories.values()
            if h is not None and 'obj' in h
        )

        bottom = data_min * 0.5
        top = max(obj_bound) * 1.5
        ax3.set_ylim(bottom=bottom, top=top)
    else:
        ax3.text(0.5, 0.5, 'Add track_history=True\nto StructuralMechanics class\nfor convergence data',
                ha='center', va='center', fontsize=12, transform=ax3.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
        ax3.set_xlim(0, 1); ax3.set_ylim(0, 1); ax3.axis('off')
    
    # Plot 4: Gradient Norm Convergance
    ax4 = axes[1, 1]
    any_grad = any(h is not None and 'grad' in h and len(h.get('grad', [])) > 0 
                   for h in histories.values())
    if any_grad:
        for name, hist in histories.items():
            #if hist is not None and len(hist.get('grad', [])) > 0:
            if True:
                grad_vals = hist['grad']
                ax4.semilogy(range(len(grad_vals)), grad_vals, 
                            label=name, linewidth=2, alpha=0.8, marker='o', markersize=3)
        ax4.axhline(y=epsilon, color='gray', linestyle='--', label=r'$\varepsilon$={}'.format(epsilon), linewidth=1)
        ax4.set_xlabel('Iteration'); ax4.set_ylabel(r'$\|\nabla E(u_k)\|$')
        ax4.set_title('(d) Convergance of Gradient Norm'); ax4.legend(); ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, 'Add track_history=True\nto StructuralMechanics class\nfor gradient data', 
                ha='center', va='center', fontsize=12, transform=ax4.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
        ax4.set_xlim(0, 1); ax4.set_ylim(0, 1); ax4.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'line_search_exp_n{n}.png', dpi=150)
    plt.show()
    print(f"\nSaved: line_search_exp_n{n}.png")

# Run
if __name__ == "__main__":
    GD_method_comparison(n=20)

### 1.4. Convergance Analysis & Conditioning

In [ ]:
def condition_number_verification(n = 3) -> StructuralMechanics:
    """Verify condition number for small n.

    Condition Number Definition:
    => kappa(E) := lambda_max / lambda_min = eta / mu

    Args:
        n (int, optional): Size of experiment n. Defaults to 3.

    Returns:
        StructuralMechanics: The problem class, if it is well conditions
    
    
    
    For a symmetric positive definite matrix, this determines:
    1. Convergence rate of gradient descent: O((kappa - 1)/(kappa + 1))^k
    2. Numerical stability of solving linear systems
    3. Sensitivity to perturbations
    """
    sm = StructuralMechanics(n = n)
    print("~~" * 67)
    print(f"Condition Number Verification | n = {n}")
    print("~~" * 67)
    print(f"mu (min eigenvalue) = {sm.mu:.6f}")
    print(f"eta (max eigenvalue) = {sm.eta:.6f}")
    print(f"kappa = eta / mu = {sm.eta/sm.mu:.4f}")
    print(f"kappa attribute = {sm.kappa:.4f}")
    print(f"Verification: {'✔️ PASSED' if abs(sm.kappa - sm.eta/sm.mu) < 1e-8 else '❌️ FAIL'}")
    print(f"Problem is {'well-conditioned' if sm.kappa<=100 else {'moderetly-conditioned' if sm.kappa<=1000 else 'ill-conditioned'}}.")
    return sm

def condition_number_experiments(max_iter: int = 10000, epsilon: float = 1e-8):
    """Complete experiments analyzing condition number vs dimension n.

    Args:
        max_iter (int, optional): Maximum number of iterations. Defaults to 10000.
        epsilon (float, optional): Error margin for the gradient norm. Defaults to 1e-8.

    Returns:
        Tuple[]: Results, and the two axis for the figure plots.
    """    
    
    # Note: N beyond ~100 becomes computationally expensive for exhaustive testing.
    N: List[int] = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    # N: List[int] = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200]
    
    results = {
        'n': [],
        'kappa': [],
        'fixed_iters': [],
        'exact_iters': [],
        'backtrack_iters': [],
        'diverged_fixed': [],
        'diverged_exact': [],
        'diverged_backtrack': []
    }
    
    print("~~" * 70)
    print(f"Condition Number Experiments | n = {N}")
    print("~~" * 70)
    
    for n in N:
        print(f"\n n = {n:3d}: ", end="")
        
        sm = StructuralMechanics(n = n)
        results['n'].append(n)
        results['kappa'].append(sm.kappa)
        
        ### FIXED STEP SIZE (Optimal alpha = 2/L) ###
        alpha_opt = 1 / sm.eta # gives O(1/k) bound (p. 46 frm the Lecture Notes) 
        try:
            _, k_f, _ = sm.gradient_descent(np.zeros(n), alpha_opt, epsilon, max_iter)
            results['fixed_iters'].append(k_f)
            results['diverged_fixed'].append(False)
            print(f"Fixed:{k_f:4d}", end=" ")
        except RuntimeError:
            results['fixed_iters'].append(max_iter)
            results['diverged_fixed'].append(True)
            print(f"Fixed:D", end=" ")
        
        ### EXACT LINE SEARCH ###
        try:
            _, k_e, _ = sm.GD_exact_line_search(np.zeros(n), epsilon, max_iter)
            results['exact_iters'].append(k_e)
            results['diverged_exact'].append(False)
            print(f"Exact:{k_e:4d}", end=" ")
        except RuntimeError:
            results['exact_iters'].append(max_iter)
            results['diverged_exact'].append(True)
            print(f"Exact:D", end=" ")
        
        ### BACKTRACKING LINE SEARCH ###
        try:
            _, k_b, _ = sm.GD_inexact_line_search(np.zeros(n), None, 0.5, 1e-4, epsilon, max_iter)
            results['backtrack_iters'].append(k_b)
            results['diverged_backtrack'].append(False)
            print(f"Bcktrk:{k_b:4d}")
        except RuntimeError:
            results['backtrack_iters'].append(max_iter)
            results['diverged_backtrack'].append(True)
            print(f"Bcktrk:D")
    
    ##### VISUALIZATION #####
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ### PLOT 1: Iterations vs n ###
    ax1 = axes[0]
    
    # data from Fixed-Step GD
    ax1.plot(results['n'], results['fixed_iters'], 
             marker='o', linewidth=2, label='Fixed (optimal)', color='#6d4aff')

    # data from Exact Line Serch GD
    ax1.plot(results['n'], results['exact_iters'], 
             marker='s', linewidth=2, label='Exact Line Search', color='#ff6b4a')
    
    # data from Backtracking GD
    ax1.plot(results['n'], results['backtrack_iters'], 
             marker='^', linewidth=2, label='Backtracking', color='#4caf50')

    # figure formatting
    ax1.axhline(y=max_iter, color='red', linestyle=':', label=f'Divergence threshold ({max_iter})')
    ax1.set_xlabel('Dimension n', fontsize=12)
    ax1.set_ylabel('Iterations', fontsize=12)
    ax1.set_title('Convergence: Iterations vs Dimension', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(left=0)
    
    ### PLOT 2: Condition Number vs n ###
    ax2 = axes[1]

    # kappa markers
    ax2.plot(results['n'], results['kappa'], 
             marker='d', linewidth=2, markersize=8, color='#6d4aff')
    
    # colour area under graph
    ax2.fill_between(results['n'], results['kappa'], alpha=0.3, color='#6d4aff')

    # figure formatting
    ax2.set_xlabel('Dimension n', fontsize=12)
    ax2.set_ylabel(r'Condition Number $\kappa = \eta / \mu$', fontsize=12)
    ax2.set_title('Condition Number Scaling with Dimension', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(left=0)
    
    # add kappa values as annotations
    for i, (n, kappa) in enumerate(zip(results['n'], results['kappa'])):
        if i % 2 == 0:  # Label every other point
            ax2.annotate(f'{kappa:.1f}', (n, kappa), 
                        textcoords="offset points", xytext=(0,10), 
                        ha='center', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(f'condition_number_analysis_n{max(N)}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved: condition_number_analysis_n{max(N)}.png")
    
    ### THEORETICAL ANALYSIS ###
    print("\n" + "~~" * 67)
    print("THEORETICAL ANALYSIS")
    print("~~" * 67)
    
    # Fit kappa ~ O(n^2)
    log_n = np.log(results['n'])
    log_kappa = np.log(results['kappa'])
    slope = np.polyfit(log_n, log_kappa, 1)[0]
    
    print(f"Log-log fit: log(kappa) approx {slope:.3f} * log(n)$")
    print(f"This suggests kappa proportional to n^{slope:.2f}")
    print("(Expected: kappa proportional to n^2)")
    
    # Convergence rate prediction (Thm 3.3)
    avg_kappa = np.mean(results['kappa'])
    slope_theory = np.log(1 - 1 / avg_kappa) # m_theory (p. 53 in the Lecture Notes)
    print(f"\nAverage condition number: avg_kappa = {avg_kappa:.2f}")
    #print(f"Theoretical convergence rate: ((kappa - 1)/(kappa + 1)) = {predicted_rate:.6f}")
    #expected_iter = int(np.ceil(np.log(1e-8)/np.log(predicted_rate)))
    #print(f"For varepsilon=10^(-8)$, expected iterations: lceil ln(varepsilon)/ln(rate) \rceil = {expected_iter}")    
    return results, (ax1, ax2)


# Combined visualization with dual y-axis (iterations + kappa)
def combined_method_visualization(max_iter: int = 10000, epsilon: float = 1e-8):
    """
    Single row visualization showing iterations (histograms) and κ (line) for each method.
    Subplots titled: (a) Fixed-Step Size, (b) Exact Line Search, (c) Backtracking Line Search
    """
    
    N: List[int] = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    methods_data = {}
    kappa_values = []
    
    for n in N:
        sm = StructuralMechanics(n=n)
        kappa_values.append(sm.kappa)

        # define instances of each method per iteration over n
        for method_name, method_func in [
            ('Fixed', lambda sm: sm.gradient_descent(np.zeros(sm.n), 1/sm.eta, epsilon, max_iter)),
            ('Exact', lambda sm: sm.GD_exact_line_search(np.zeros(sm.n), epsilon, max_iter)),
            ('Backtracking', lambda sm: sm.GD_inexact_line_search(np.zeros(sm.n), None, 0.5, 1e-4, epsilon, max_iter))
        ]:
            if method_name not in methods_data:
                methods_data[method_name] = {'n': [], 'iter': []}
            
            try:
                _, k, _ = method_func(sm)
                methods_data[method_name]['n'].append(n)
                methods_data[method_name]['iter'].append(k)
            except:
                methods_data[method_name]['n'].append(n)
                methods_data[method_name]['iter'].append(max_iter)
    
    # Create figure with 3 subplots in a single row for the three method experiments
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    method_titles = ['Fixed', 'Exact', 'Backtracking']
    subplot_labels = ['(a)', '(b)', '(c)']
    
    for idx, (method_name, data) in enumerate(methods_data.items()):
        ax = axes[idx]
        
        # Histogram-style bars for iterations (using bar plot)
        bar_width = 3.0  # Width of each bar
        x_positions = np.arange(len(data['n']))
        ax.bar(x_positions, data['iter'], width=bar_width, 
              color=plt.cm.tab10(idx), alpha=0.7, label='Iterations')
        
        # Overlay line connecting iteration values
        #ax.plot(x_positions, data['iter'], 
        #       marker='o', linewidth=2, markersize=6, 
        #       label='Iterations (trend)', color='black', alpha=0.8)
        
        # right axis: Condition Number as line graph
        ax2 = ax.twinx()
        ax2.plot(x_positions, kappa_values, 
                marker='s', linewidth=2, markersize=8,
                label='Condition Number kappa', color='gray', alpha=0.8, linestyle='-')
        ax2.set_ylabel('Condition Number kappa', fontsize=11, color='gray')
        ax2.tick_params(axis='y', labelcolor='gray', labelsize=10)
        ax2.set_ylim(bottom=0)
        
        # set labels and ticks
        ax.set_xticks(x_positions)
        ax.set_xticklabels([str(n) for n in data['n']], fontsize=10)
        ax.set_ylabel('Iterations', fontsize=11)
        
        # combined legend
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
        
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_xlim(left=-1, right=len(N))
        
        # add iteration value labels on top of bars
        for i, val in enumerate(data['iter']):
            ax.annotate(str(val), (i, val), 
                       textcoords="offset points", xytext=(0,5), 
                       ha='center', fontsize=7, alpha=0.7)
    
    # main title with subtitles
    fig.suptitle('Method-Specific Analysis: Iterations + Condition Number', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    # individual subplot subtitles
    for idx, method_title in enumerate(method_titles):
        subtitle = f"{subplot_labels[idx]} {method_title}-Step Size" if method_title != 'Exact' else \
                   f"{subplot_labels[idx]} {method_title} Line Search"
        axes[idx].set_title(subtitle, fontsize=12, fontweight='normal', pad=15)
    
    axes[2].set_xlabel('Dimension n', fontsize=11)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Make room for suptitle
    plt.savefig(f'combined_method_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Saved: combined_method_visualization.png")

# Run verification first
if __name__ == "__main__":
    print("\n" + "~~" * 67)
    print("STEP 1: VERIFY CONDITION NUMBER FOR n=3")
    print("~~" * 67)
    sm_test = condition_number_verification(n=3)
    
    print("\n" + "~~" * 67)
    print("STEP 2: CONDITION NUMBER EXPERIMENTS ACROSS DIMENSIONS")
    print("~~" * 67)
    results, axes = condition_number_experiments(max_iter=600000, epsilon=1e-8)
    
    print("\n" + "~~" * 67)
    print("COMBINED METHOD-SPECIFIC VISUALIZATION")
    print("~~" * 67)
    print("Run combined_method_visualization() for per-method breakdown")
    
    # Uncomment to run combined view (NB: it takes a while ~ 15 min)
    combined_method_visualization(max_iter=600000, epsilon=1e-8)

### 1.5. Heavy-Ball Momentum Method & Nesterov's Accelerated Gradient Method Comparison

In [ ]:
def momentum_methods_comparison(n_values: List[int] = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                                 max_iter: int = 10000, 
                                 epsilon: float = 1e-8) -> Dict:
    """
    Comprehensive comparison of Heavy-Ball and Nesterov accelerated gradient methods.
    
    Tracks characteristics specific to momentum methods:
    - Convergence iterations vs condition number kappa
    - Actual vs theoretical convergence rates
    - Momentum coefficient beta scaling
    - Objective and gradient norm decay
    
    References Lecture Notes Chapter 4 (Advanced Methods):
    - Theorem 4.2: Heavy-ball local convergence
    - Theorem 4.4: Nesterov global convergence
    """
    
    # Storage for results
    results = {
        'n': [],
        'kappa': [],
        'gd_iters': [],
        'hb_iters': [],
        'nesterov_iters': [],
        'gd_theoretical_q': [],
        'hb_theoretical_q': [],
        'nesterov_theoretical_q': [],
        'hb_beta': [],
        'nesterov_beta': [],
        'converged_hb': [],
        'converged_nesterov': []
    }
    
    print("~~" * 70)
    print("Momentum Methods Comparison | Heavy-Ball vs Nesterov")
    print("~~" * 70)
    
    for n in n_values:
        print(f"\nn={n:3d}: ", end="")
        sm = StructuralMechanics(n=n)
        
        ### BASIC PROPERTIES ###
        results['n'].append(n)
        results['kappa'].append(sm.kappa)
        
        ### GRADIENT DESCENT (OPTIMAL STEP) ###
        # Theoretical: q_GD = (\kappa - 1) / (\kappa + 1), iterations ~ O(kappa)
        alpha_gd = 2 / (sm.mu + sm.eta)
        
        # Add GD theoretical convergence factor
        q_gd_theoretical = (sm.kappa - 1) / (sm.kappa + 1)
        results['gd_theoretical_q'].append(q_gd_theoretical)
        
        try:
            _, gd_k, _ = sm.gradient_descent(np.zeros(n), alpha_gd, epsilon, max_iter)
            results['gd_iters'].append(gd_k)
            results['converged_hb'].append(False)  # placeholder
            results['converged_nesterov'].append(False)  # placeholder
            print(f"GD:{gd_k:4d}", end=" ")
        except RuntimeError:
            results['gd_iters'].append(max_iter)
            print(f"GD:D", end=" ")
        
        ### HEAVY-BALL METHOD ###
        # Polyak parameters (Thm 4.2): 
        # =>  alpha = 4/(sqrt{L} + sqrt{mu})^2
        # =>  beta = ((sqrt{L} - sqrt{mu}) / (sqrt{L} + sqrt{mu}))^2        
        # Theoretical convergence: q_HB = (sqrt{kappa} - 1) / (sqrt{kappa} + 1), iterations ~ O(sqrt{kappa})
        
        sqrt_mu = np.sqrt(sm.mu)
        sqrt_eta = np.sqrt(sm.eta)
        beta_hb = ((sqrt_eta - sqrt_mu) / (sqrt_eta + sqrt_mu)) ** 2
        alpha_hb = 4 / (sqrt_eta + sqrt_mu) ** 2
        
        results['hb_beta'].append(beta_hb)
        results['hb_theoretical_q'].append((np.sqrt(sm.kappa) - 1) / (np.sqrt(sm.kappa) + 1))
        
        try:
            u_hb, hb_k, obj_hb, hist_hb = sm.heavy_ball(
                np.zeros(n), alpha_hb, beta_hb, epsilon, max_iter, track_history=True
            )
            results['hb_iters'].append(hb_k)
            results['converged_hb'][-1] = True if len(results['converged_hb']) > 0 else True
            print(f"HB:{hb_k:4d}", end=" ")
        except RuntimeError:
            results['hb_iters'].append(max_iter)
            print(f"HB:D", end=" ")
        
        ### NESTEROV'S ACCELERATED GRADIENT METHOD ###
        # Theoretical (Thm 4.4): q_NAG = 1 - 1 / \sqrt{\kappa}, iterations ~ O(\sqrt{\kappa})
        # Parameters: \alpha = 1 / L, \beta = (\sqrt{L} - \sqrt{\mu}) / (\sqrt{L} + \sqrt{\mu})
        beta_nesterov = (sqrt_eta - sqrt_mu) / (sqrt_eta + sqrt_mu)
        alpha_nesterov = 1 / sm.eta
        
        results['nesterov_beta'].append(beta_nesterov)
        results['nesterov_theoretical_q'].append(1 - 1/np.sqrt(sm.kappa))
        
        try:
            u_nest, nest_k, obj_nest, hist_nest = sm.nesterov(
                np.zeros(n), alpha_nesterov, beta_nesterov, epsilon, max_iter, track_history=True
            )
            results['nesterov_iters'].append(nest_k)
            results['converged_nesterov'][-1] = True if len(results['converged_nesterov']) > 0 else True
            print(f"Nest:{nest_k:4d}")
        except RuntimeError:
            results['nesterov_iters'].append(max_iter)
            print(f"Nest:D")
    
    ##### VISUALIZATION #####
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    ### PLOT 1: ITERATIONS VS CONDITION NUMBER ###
    ax1 = axes[0, 0]
    ax1.plot(results['n'], results['gd_iters'], 'o-', linewidth=2, markersize=6,
            label='Gradient Descent', color='#6d4aff')
    ax1.plot(results['n'], results['hb_iters'], 's-', linewidth=2, markersize=6,
            label='Heavy-Ball', color='#ff6b4a')
    ax1.plot(results['n'], results['nesterov_iters'], '^-', linewidth=2, markersize=6,
            label='Nesterov', color='#4caf50')
    
    ax1.axhline(y=max_iter, color='red', linestyle=':', label=f'Divergence ({max_iter})')
    ax1.set_xlabel('Dimension n', fontsize=12)
    ax1.set_ylabel('Iterations', fontsize=12)
    ax1.set_title('(a) Convergence: Iterations vs Dimension', fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(left=0)
    
    ### PLOT 2: CONDITION NUMBER SCALING ###
    ax2 = axes[0, 1]
    ax2.plot(results['n'], results['kappa'], 'd-', linewidth=2, markersize=8,
            label=r'Condition Number $\kappa$', color='#6d4aff')
    ax2.fill_between(results['n'], results['kappa'], alpha=0.3, color='#6d4aff')

    # Add theoretical scaling reference line (\kappa \propto n^2)
    n_arr = np.array(results['n'])
    
    # Anchor the reference curve to the first data point, then scale as n^2
    kappa_ref = results['kappa'][0] * (n_arr / n_arr[0]) ** 2

    ax2.plot(results['n'], kappa_ref, '--', color='gray', alpha=0.5,
            label=r'$\kappa \propto n^2$ (theoretical)', linewidth=1)    
    ax2.set_xlabel('Dimension n', fontsize=12)
    ax2.set_ylabel(r'Condition Number $\kappa$', fontsize=12)
    ax2.set_title('(b) Conditioning', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(left=0)
    
    ### PLOT 3: CONVERGENCE FACTORS ###
    ax3 = axes[0, 2]
    ax3.plot(results['n'], results['gd_theoretical_q'], 'o-', linewidth=2,
            label=r'GD: $(\kappa - 1) / (\kappa + 1)$', color='#6d4aff')
    ax3.plot(results['n'], results['hb_theoretical_q'], 's-', linewidth=2,
            label=r'HB: $(\sqrt{\kappa} - 1) / (\sqrt{\kappa} + 1)$', color='#ff6b4a')
    ax3.plot(results['n'], results['nesterov_theoretical_q'], '^-', linewidth=2,
            label=r'Nesterov: $1 - 1/\sqrt{\kappa}$', color='#4caf50')
    
    ax3.axhline(y=1, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax3.set_xlabel('Dimension n', fontsize=12)
    ax3.set_ylabel('Contraction Factor q', fontsize=12)
    ax3.set_title('(c) Theoretical Convergence Rates', fontsize=13, fontweight='bold')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(bottom=0.75, top=1.05)
    ax3.set_xlim(left=0)
    
    ### PLOT 4: MOMENTUM COEFFICIENTS ###
    ax4 = axes[1, 0]
    ax4.plot(results['n'], results['hb_beta'], 's-', linewidth=2, markersize=6,
            label=r'Heavy-Ball $\beta$', color='#ff6b4a')
    ax4.plot(results['n'], results['nesterov_beta'], '^-', linewidth=2, markersize=6,
            label=r'Nesterov $\beta$', color='#4caf50')
    
    ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax4.set_xlabel('Dimension n', fontsize=12)
    ax4.set_ylabel(r'Momentum Parameter $\beta$', fontsize=12)
    ax4.set_title('(d) Momentum Coefficients', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(bottom=0.65, top=1.05)
    ax4.set_xlim(left=0)
    
    # Add theoretical limit annotation
    max_kappa = max(results['kappa'])
    asymptote_hb = ((np.sqrt(max_kappa) - 1) / (np.sqrt(max_kappa) + 1)) ** 2
    ax4.axhline(y=asymptote_hb, color='orange', linestyle=':', alpha=0.5,
               label=r'HB $\beta \to {(np.sqrt(max_kappa)-1)/(np.sqrt(max_kappa)+1):.3f}^2$', linewidth=1)
    
    ### PLOT 5: SPEEDUP FACTOR ###
    ax5 = axes[1, 1]
    speedup_hb = np.array(results['gd_iters']) / np.maximum(np.array(results['hb_iters']), 1)
    speedup_nest = np.array(results['gd_iters']) / np.maximum(np.array(results['nesterov_iters']), 1)
    
    ax5.plot(results['n'], speedup_hb, 's-', linewidth=2, markersize=6,
            label='HB Speedup over Fixed-Step GD', color='#ff6b4a')
    ax5.plot(results['n'], speedup_nest, '^-', linewidth=2, markersize=6,
            label='Nesterov Speedup over Fixed-Step GD', color='#4caf50')
    
    ax5.axhline(y=1, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax5.set_xlabel('Dimension n', fontsize=12)
    ax5.set_ylabel('Speedup Factor (GD/Momentum)', fontsize=12)
    ax5.set_title('(e) Acceleration Relative to GD', fontsize=13, fontweight='bold')
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)
    ax5.set_xlim(left=0)
    
    ### PLOT 6: OBJECTIVE CONVERGENCE (single n example) ###
    ax6 = axes[1, 2]
    
    # Run single instance for convergence curves
    n_example = n_values[len(n_values)//2]  # Middle value
    sm_example = StructuralMechanics(n=n_example)
    
    alpha_gd_ex = 2 / (sm_example.mu + sm_example.eta)
    alpha_hb_ex = 4 / (np.sqrt(sm_example.eta) + np.sqrt(sm_example.mu)) ** 2
    beta_hb_ex = ((np.sqrt(sm_example.eta) - np.sqrt(sm_example.mu)) / 
                  (np.sqrt(sm_example.eta) + np.sqrt(sm_example.mu))) ** 2
    alpha_nest_ex = 1 / sm_example.eta
    beta_nest_ex = (np.sqrt(sm_example.eta) - np.sqrt(sm_example.mu)) / \
                   (np.sqrt(sm_example.eta) + np.sqrt(sm_example.mu))

    # Analytical minimiser and minimum value
    u_example_analytical = LA.solve(sm_example.K, sm_example.b)
    obj_example_analytical = sm_example._objective(u_example_analytical)

    try:
        _, _, _, hist_gd = sm_example.gradient_descent(np.zeros(n_example), alpha=alpha_gd_ex,
                                                    epsilon=epsilon, max_iter=max_iter, track_history=True)
        if hist_gd:
            ax6.semilogy(range(len(hist_gd['obj'])), 
                        [abs(o - obj_example_analytical) for o in hist_gd['obj']],
                        '-', linewidth=2, label='GD', color='#6d4aff', alpha=0.7)
    except:
        pass
    
    try:
        _, _, _, hist_hb = sm_example.heavy_ball(np.zeros(n_example), alpha=alpha_hb_ex, beta=beta_hb_ex,
                                                epsilon=epsilon, max_iter=max_iter, track_history=True)
        if hist_hb:
            ax6.semilogy(range(len(hist_hb['obj'])), 
                        [abs(o - obj_example_analytical) for o in hist_hb['obj']],
                        '-', linewidth=2, label='Heavy-Ball', color='#ff6b4a', alpha=0.7)
    except:
        pass
    
    try:
        _, _, _, hist_nest = sm_example.nesterov(np.zeros(n_example), alpha=alpha_nest_ex, beta=beta_nest_ex,
                                                epsilon=epsilon, max_iter=max_iter, track_history=True)
        if hist_nest:
            ax6.semilogy(range(len(hist_nest['obj'])), 
                        [abs(o - obj_example_analytical) for o in hist_nest['obj']],
                        '-', linewidth=2, label='Nesterov', color='#4caf50', alpha=0.7)
    except:
        pass
    
    ax6.set_xlabel('Iteration', fontsize=12)
    ax6.set_ylabel(r'$\mid f(x) - f^* \mid$', fontsize=12)
    ax6.set_title(r'(f) Objective Decay (n={}, $\kappa$={:.1f})'.format(n_example, sm_example.kappa), 
                 fontsize=13, fontweight='bold')
    ax6.set_ylim(bottom = 1e-8)
    ax6.set_xlim(right=35000)
    ax6.legend(fontsize=9)
    ax6.grid(True, alpha=0.3)
    
    plt.suptitle('Momentum Methods: Heavy-Ball vs Nesterov Acceleration', 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'momentum_methods_comparison_n{max(n_values)}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved: momentum_methods_comparison_n{max(n_values)}.png")
    
    ##### THEORETICAL VERIFICATION #####
    print("\n" + "~~" * 70)
    print("THEORETICAL VERIFICATION")
    print("~~" * 70)
    
    kappa_vals = np.array(results['kappa'])
    gd_speedup = np.array(results['gd_iters']) / np.maximum(np.array(results['hb_iters']), 1)
    
    print(f"\nAverage condition number: kappa_avg= {np.mean(kappa_vals):.2f}")
    print(f"Max condition number: kappa_max = {np.max(kappa_vals):.2f}")
    
    # Verify O(\sqrt{\kappa}) vs O(\kappa) scaling
    kappa_array = np.array(results['kappa'])
    mask = kappa_array < kappa_array.max()

    log_kappa = np.log(kappa_vals[mask])
    log_gd = np.log(np.array(results['gd_iters'])[mask])
    log_hb = np.log(np.array(results['hb_iters'])[mask])
    
    gd_slope = np.polyfit(log_kappa, log_gd, 1)[0]
    hb_slope = np.polyfit(log_kappa, log_hb, 1)[0]
    
    print(f"\nGD iteration scaling: iter prop to kappa^{gd_slope:.2f}")
    print(f"Expected: O(kappa), Actual: O(kappa^{gd_slope:.2f})")
    
    print(f"\nHB iteration scaling: iter prop to kappa^{hb_slope:.2f}")
    print(f"Expected: O(sqrt(kappa)), Actual: O(kappa^{hb_slope:.2f})")
    
    # Convergence factor verification
    avg_kappa = np.mean(kappa_vals)
    q_gd_th = (avg_kappa - 1) / (avg_kappa + 1)
    q_hb_th = (np.sqrt(avg_kappa) - 1) / (np.sqrt(avg_kappa) + 1)
    q_nest_th = 1 - 1/np.sqrt(avg_kappa)
    
    print(f"\nTheoretical convergence factors (kappa={avg_kappa:.1f}):")
    print(f"  GD: q = {q_gd_th:.4f}")
    print(f"  HB: q = {q_hb_th:.4f}")
    print(f"  Nesterov: q = {q_nest_th:.4f}")
    
    print(f"\nAcceleration ratio: q_GD/q_HB = {q_gd_th/q_hb_th:.2f}x better convergence factor")
    print(f"(Lecture Thm 4.2, 4.4 verified: momentum methods achieve O(sqrt(kappa)) vs O(kappa))")
    
    return results, fig

# Run comparison
if __name__ == "__main__":
    print("~~" * 70)
    print("ACCELERATED METHODS COMPARISON EXPERIMENT")
    print("Verifying Lecture Notes Chapter 4 (Thm 4.2, 4.4)")
    print("~~" * 70)
    
    results, fig = momentum_methods_comparison(
        n_values=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
        max_iter=100000,
        epsilon=1e-8
    )
    
    print("\n" + "~~" * 70)
    print("Experiment complete. Review plots for theoretical verification.")
    print("~~" * 70)

### 1.6. Newton's Method Comparison

In [ ]:
def newton_full_comparison(n_values: List[int] = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                           max_iter: int = 100000,
                           epsilon: float = 1e-8,
                           return_dict: Dict = None
                           ):
    """Final comparison: Newton's method vs all first-order methods.

    Args:
        n_values (List[int], optional): Value of n for which to run experiment. Defaults to [10, 20, ..., 100].
        max_iter (int, optional): Maximum number of iterations. Defaults to 100000.
        epsilon (float, optional): Error margin. Defaults to 1e-8.
        return_dict (Dict, optional): _description_. Defaults to None.

    """    
    """
    
    
    Tracks characteristics specific to this comparison:
    - Iteration counts (algorithmic efficiency)
    - Wall-clock time (practical efficiency incl. per-iteration cost)
    - Cost per iteration analysis (O(n²) matvec vs O(n³) linear solve)
    
    References Lecture Notes Chapter 4:
    - Prop. 4.2: Newton converges in ONE step for quadratic f
    - Sec. 4.6: Limitations - Newton step costs O(n³) per iteration
    - Thm 4.2/4.4: HB and Nesterov achieve O(√κ) iterations
    """
    
    results = {
        'n': [], 'kappa': [],
        'gd_iters': [], 'exact_iters': [], 'bt_iters': [],
        'hb_iters': [], 'nest_iters': [], 'newton_iters': [],
        'gd_time': [], 'exact_time': [], 'bt_time': [],
        'hb_time': [], 'nest_time': [], 'newton_time': []
    }
    
    print("=" * 70)
    print("Full Method Comparison: Newton vs All Previous Methods")
    print("=" * 70)
    print(f"{'n':>4} {'κ':>8} | {'GD':>6} {'Exact':>6} {'Bt':>6} {'HB':>5} "
          f"{'Nest':>5} {'New':>4} | timings (s)")
    print("-" * 70)
    
    for n in n_values:
        sm = StructuralMechanics(n=n)
        u0 = np.zeros(n)
        
        # --- Optimal parameter setup ---
        alpha_gd = 2 / (sm.mu + sm.eta)
        sqrt_mu, sqrt_eta = np.sqrt(sm.mu), np.sqrt(sm.eta)
        alpha_hb = 4 / (sqrt_eta + sqrt_mu) ** 2
        beta_hb = ((sqrt_eta - sqrt_mu) / (sqrt_eta + sqrt_mu)) ** 2
        alpha_nest = 1 / sm.eta
        beta_nest = (sqrt_eta - sqrt_mu) / (sqrt_eta + sqrt_mu)
        
        results['n'].append(n)
        results['kappa'].append(sm.kappa)
        
        def timed_run(method_call, key_iters, key_time):
            try:
                t0 = time.perf_counter()
                out = method_call()
                dt = time.perf_counter() - t0
                results[key_iters].append(out[1])  # second return = iteration count
                results[key_time].append(dt)
                return out[1]
            except RuntimeError:
                results[key_iters].append(max_iter)
                results[key_time].append(float('nan'))
                return max_iter
        
        k_gd   = timed_run(lambda: sm.gradient_descent(u0, alpha_gd, epsilon, max_iter),
                           'gd_iters', 'gd_time')
        k_ex   = timed_run(lambda: sm.GD_exact_line_search(u0, epsilon, max_iter),
                           'exact_iters', 'exact_time')
        k_bt   = timed_run(lambda: sm.GD_inexact_line_search(u0, None, 0.5, 1e-4, epsilon, max_iter),
                           'bt_iters', 'bt_time')
        k_hb   = timed_run(lambda: sm.heavy_ball(u0, alpha_hb, beta_hb, epsilon, max_iter),
                           'hb_iters', 'hb_time')
        k_ne   = timed_run(lambda: sm.nesterov(u0, alpha_nest, beta_nest, epsilon, max_iter),
                           'nest_iters', 'nest_time')
        k_nw   = timed_run(lambda: sm.newton(u0, epsilon, max_iter),
                           'newton_iters', 'newton_time')
        
        print(f"{n:>4} {sm.kappa:>8.0f} | {k_gd:>6} {k_ex:>6} {k_bt:>6} {k_hb:>5} "
              f"{k_ne:>5} {k_nw:>4} | "
              f"{results['gd_time'][-1]:.2f}/{results['newton_time'][-1]*1000:.1f}ms")
    
    ##### VISUALIZATION #####
    fig, axes = plt.subplots(ncols=2, figsize=(15, 10))
    n_arr = np.array(results['n'])
    
    # (a) Iterations vs n (log scale — huge disparities!)
    ax1 = axes[0]
    data_iters = [
        ('Fixed GD', results['gd_iters'], '#6d4aff', 'o'),
        ('Exact LS', results['exact_iters'], '#3b82f6', 'v'),
        ('Backtracking', results['bt_iters'], '#9333ea', 'P'),
        ('Heavy-Ball', results['hb_iters'], '#ff6b4a', 's'),
        ('Nesterov', results['nest_iters'], '#4caf50', '^'),
        ('Newton', results['newton_iters'], '#d4af37', '*'),
    ]
    for name, ys, c, m in data_iters:
        ax1.semilogy(n_arr, np.maximum(ys, 1), m + '-', color=c, markersize=8,
                     linewidth=2, label=name, alpha=0.8)
    ax1.set_xlabel('Dimension n', fontsize=12)
    ax1.set_ylabel('Iterations (log scale)', fontsize=12)
    ax1.set_title('(a) Iterations: Newton dominates', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3, which='both')
    ax1.set_xlim(left=0)
    
    # (b) Wall-clock time vs n (the honest comparison)
    ax2 = axes[1]
    data_times = [
        ('Fixed GD', results['gd_time'], '#6d4aff', 'o'),
        ('Heavy-Ball', results['hb_time'], '#ff6b4a', 's'),
        ('Nesterov', results['nest_time'], '#4caf50', '^'),
        ('Newton', results['newton_time'], '#d4af37', '*'),
    ]
    for name, ys, c, m in data_times:
        ax2.plot(n_arr, np.array(ys) * 1000, m + '-', color=c, markersize=8,
                linewidth=2, label=name, alpha=0.8)
    ax2.set_xlabel('Dimension n', fontsize=12)
    ax2.set_ylabel('Wall-clock time (ms)', fontsize=12)
    ax2.set_title('(b) Runtime: per-iteration cost matters', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
    ax2.set_xlim(left=0)
    
    plt.suptitle('Newton vs First-Order Methods: Speed vs Cost Trade-off',
                 fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('newton_full_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results, fig

# Run
if __name__ == "__main__":
    results_final, fig_final = newton_full_comparison(
        n_values=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
        max_iter=100000,
        epsilon=1e-8
    )